# Notebook 1: Data Download and Quality Checks

This notebook builds the daily price data used throughout the project. It downloads prices for the 16 ETFs in the two study universes from Yahoo Finance, stores a frozen copy of the download, checks the data for errors, and saves a clean daily price table that Notebook 2 uses to build features and labels.

**Work packages:** WBS 2.2 (Download and freeze) and 2.3 (Quality checks and cleaning)

**Data:** daily open, high, low, close, adjusted close, volume, dividends and splits for 16 ETFs, from 2 January 2009 to 30 September 2026. The extra year before 2010 is needed so that every indicator has a full history at the first monthly observation in January 2010.

**Outputs** (Google Drive, `Capstone/data`):

| File | Folder | Contents |
|---|---|---|
| Raw Prices.csv | raw | The download exactly as received from Yahoo |
| Raw Prices Info.json | raw | Download date, library versions and checksum |
| Quality Report.csv | reports | Result of every quality check |
| Daily Prices.parquet / .csv | processed | Clean daily prices, the input to Notebook 2 |
| Coverage.csv | reports | One summary row per ETF |

**Why the download is frozen:** Yahoo recalculates adjusted prices each time a fund pays a dividend, so downloads made on different days give slightly different numbers. The data is therefore downloaded once and saved with a checksum, and every later step reads that saved copy. This keeps all results reproducible. Running the notebook again does not download new data.

**How to run:** Runtime > Run all. A full run takes about two minutes.

## 1. Setup

Connects Google Drive, where all data files are stored, and loads the project code from the GitHub repository. Run this cell first in every new Colab session. When Colab asks for access, choose the Google account that holds the `Capstone/data` folder.

In [ ]:
# Connect Google Drive. All data files are read from and saved to
# MyDrive/Capstone/data.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

# Download the project code on the first run of a session; on later runs,
# update it so any changes pushed to GitHub are picked up.
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

# Install the pinned library versions listed in requirements.txt.
%pip install --quiet -r {REPO_DIR}/requirements.txt

# Make the project modules importable with "from src import ...".
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from src import config, download, cleaning

# Reload the modules so code changes pulled from GitHub take effect
# without restarting the Colab runtime.
for module in (config, download, cleaning):
    importlib.reload(module)

# Folder paths on Google Drive (created if they do not exist yet).
DIRS = config.data_dirs()

# One-time housekeeping: files saved under the earlier naming scheme are
# renamed (the frozen download) or moved to data/archive (everything else).
# Does nothing once the files have been renamed.
download.rename_old_files(DIRS)

print("Tickers:", len(config.ALL_TICKERS), config.ALL_TICKERS)
print("Period: ", config.START, "to", config.END, "(end date not included)")
print("yfinance version:", yf.__version__)

## 2. Download and freeze

Downloads the prices from Yahoo Finance and saves them as `Raw Prices.csv`, together with an information file that records the download date, the library versions and a checksum of the file. This step runs only when no frozen download exists. To replace the frozen download on purpose, set `NEW_DOWNLOAD = True`, run the cell once, and set it back to `False`.

In [ ]:
NEW_DOWNLOAD = False

if download.is_frozen(DIRS["raw"]) and not NEW_DOWNLOAD:
    print("Frozen download found, no new download needed:", config.FILES["raw"])
else:
    # Request all 16 tickers; tickers that Yahoo drops are requested again.
    prices_raw = download.download_prices()
    # Save the data and its checksum. An existing file is only replaced
    # when NEW_DOWNLOAD is True.
    download.freeze(prices_raw, DIRS["raw"], overwrite=NEW_DOWNLOAD)

## 3. Load the frozen download

Reads `Raw Prices.csv` and recomputes its checksum. If the checksum differs from the one recorded at download time, the file has been changed since it was frozen and the notebook stops.

In [ ]:
prices, info = download.load_frozen(DIRS["raw"])
print("Downloaded on:", info["downloaded_utc"], "with yfinance", info["versions"]["yfinance"])

# One row per ETF per trading day.
prices.head()

## 4. Quality checks

Runs about thirteen checks on every ETF: full coverage of the period, missing or duplicate days, non-positive prices, one-day moves above 25%, changes in the adjustment factor away from dividend dates, days with zero volume, and repeated identical prices.

Each check returns **PASS**, **WARN** (handled by the cleaning step, or worth reviewing once) or **FAIL** (the data cannot be used until the problem is fixed, usually by downloading again). The notebook stops on any FAIL. Only the lines that are not PASS are printed.

In [ ]:
report = cleaning.run_quality_checks(prices)
overall = cleaning.summarize(report)

# Keep the full report, including passed checks, for the process log.
report.to_csv(DIRS["reports"] / config.FILES["quality"], index=False)

if overall == "FAIL":
    raise RuntimeError("A quality check failed. Fix the problem before cleaning.")

## 5. Clean and save

Puts all ETFs on the same trading calendar, fills a single missing day with the previous day's price (every fill is logged), and marks days with zero volume. The cleaned table is then checked for a price at every month-end, since the monthly data in Notebook 2 is sampled on those dates.

In [ ]:
clean, fill_log = cleaning.clean_prices(prices)

# Every ETF needs a price on all 213 month-ends from January 2009 to
# September 2026.
month_end_report = cleaning.check_month_ends(clean)
cleaning.summarize(month_end_report)
if (month_end_report["status"] == "FAIL").any():
    raise RuntimeError("A month-end price is missing.")

# Save as Parquet (fast to load, keeps data types) and CSV (easy to open).
cleaning.save_clean(clean, fill_log, DIRS["processed"], DIRS["reports"])
print(f"Prices filled from the previous day: {len(fill_log)}")

# Summary per ETF: dates covered, filled days, zero-volume days, dividends
# and total return over the full period.
coverage = cleaning.coverage_table(clean)
coverage.to_csv(DIRS["reports"] / config.FILES["coverage"], index=False)
coverage

## 6. Visual check: growth of $1

Shows how $1 invested on 2 January 2009 grows in each ETF, on a log scale. Expected patterns: the March 2020 crash in every equity fund, a nearly flat line for bonds (AGG) and a volatile line for commodities (DBC). A sudden vertical jump that does not match a real market event would point to a faulty price adjustment.

In [ ]:
adj = clean.pivot(index="date", columns="ticker", values="adj_close")
growth = adj / adj.iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (name, tickers) in zip(axes, [("ETF_A", config.ETF_A), ("ETF_B", config.ETF_B)]):
    growth[tickers].plot(ax=ax, logy=True, linewidth=1)
    ax.set_title(f"{name}: growth of $1 since 2 January 2009")
    ax.set_xlabel("")
    ax.legend(ncol=2, fontsize=8)
axes[0].set_ylabel("value of $1 (log scale)")
plt.tight_layout()
plt.show()

## 7. Cross-check with a second source (optional)

Compares monthly returns for SPY, AGG and JPXN with prices from Stooq. The check runs only if the files `Stooq SPY.csv`, `Stooq AGG.csv` or `Stooq JPXN.csv` have been placed in `data/raw`. Monthly returns that differ by more than 0.1 percentage point are flagged.

In [ ]:
rows = []
for ticker in ["SPY", "AGG", "JPXN"]:
    path = DIRS["raw"] / f"Stooq {ticker}.csv"
    if path.exists():
        rows.append(cleaning.cross_check(clean, path, ticker))

if rows:
    display(pd.DataFrame(rows))
else:
    print("No Stooq files found, cross-check skipped.")

## 8. Summary

Key figures for the process log and the Data Specification sheet. After the run, save the notebook (File > Save) so the outputs are kept.

In [ ]:
print(f"Raw file:      {config.FILES['raw']}")
print(f"Checksum:      {info['md5']}")
print(f"Downloaded:    {info['downloaded_utc']}")
print(f"Rows:          {len(clean):,}")
print(f"Trading days:  {clean['date'].nunique():,}")
print(f"Month-ends:    {len(cleaning.month_end_dates(clean))}")
print(f"Checks:        {overall}, prices filled: {len(fill_log)}")